In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("orvile/x-ray-baggage-anomaly-detection")

print("Path to dataset files:", path)

In [ ]:
# Install YOLOv5 dependencies
!pip install -q ultralytics

import torch
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
import glob

# Check if CUDA is available and set device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Print torch version and CUDA availability for debugging
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
yolo_config_file = "/kaggle/input/x-ray-baggage-anomaly-detection/data.yaml"

In [ ]:
# Train YOLOv8 model
def train_yolo_model(config_file, model_size='s', epochs=30, batch_size=16, image_size=640):
    # Initialize the YOLO model with pre-trained weights
    model = YOLO(f'yolov8{model_size}.pt')  # Use YOLOv8 small model
    
    # Train the model
    results = model.train(
        data=config_file,
        epochs=epochs,
        imgsz=image_size,
        batch=batch_size,
        name='xray_baggage_detector',
        degrees=15,
        flipud=0.5,
        fliplr=0.5,
        lr0=0.001,
        patience=5,
        cache=True,
        optimizer='Adam',
        workers=4,
        verbose=True,
        device=0 if torch.cuda.is_available() else 'cpu'
    )
    
    return model, results

# Set training parameters
EPOCHS = 15  
BATCH_SIZE = 8 
IMAGE_SIZE = 640  

try:
    # Train the model
    print("Starting model training...")
    model, results = train_yolo_model(
        config_file=yolo_config_file,
        model_size='s',
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE
    )
    print("Training completed successfully!")
except Exception as e:
    print(f"Error during training: {e}")

In [ ]:
import pandas as pd


def plot_training_results_from_csv(run_dir):
    results_csv = os.path.join(run_dir, 'results.csv')
    if not os.path.exists(results_csv):
        print(f"No results.csv found in {run_dir}")
        return

    # Load results.csv
    results = pd.read_csv(results_csv)

    # Plot selected metrics
    metrics = ['train/box_loss', 'train/obj_loss', 'val/box_loss', 'val/obj_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']

    plt.figure(figsize=(20, 15))
    for i, metric in enumerate(metrics):
        if metric in results.columns:
            plt.subplot(4, 2, i+1)
            plt.plot(results[metric], label=metric)
            plt.title(metric)
            plt.xlabel('Epoch')
            plt.legend()
            plt.grid()
    plt.tight_layout()
    plt.show()

plot_training_results_from_csv('/kaggle/working/runs/detect/xray_baggage_detector/') 

In [ ]:
# Evaluate the trained model
def evaluate_model(model):
    try:
        # Validate on the validation set
        val_results = model.val()
        
        print("\nValidation Results:")
        print(f"mAP@0.5: {val_results.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {val_results.box.map:.4f}")
        
        # Print per-class metrics
        print("\nPer-class Performance:")
        for i, cls_name in enumerate(model.names):
            if i < len(val_results.box.maps):
                print(f"Class '{cls_name}': mAP@0.5 = {val_results.box.maps[i]:.4f}")
        
        return val_results
    except Exception as e:
        print(f"Error during evaluation: {e}")

# Evaluate the model if available
if 'model' in locals():
    val_results = evaluate_model(model)

In [ ]:
dataset_path = path 
# Function to run inference on sample images
def test_inference(model, test_img_dir, num_samples=5):
    # Find the test directory if it doesn't exist at the expected location
    if not os.path.exists(test_img_dir):
        test_dirs = glob.glob(os.path.join(dataset_path, '**', 'test', '**', 'images'), recursive=True)
        if test_dirs:
            test_img_dir = test_dirs[0]
        else:
            print(f"Test image directory not found at {test_img_dir}")
            return
    
    # Get sample images from test directory
    test_images = glob.glob(os.path.join(test_img_dir, "*.jpg")) + glob.glob(os.path.join(test_img_dir, "*.png"))
    
    if not test_images:
        print(f"No test images found in {test_img_dir}")
        return
    
    # Select random samples
    import random
    samples = random.sample(test_images, min(num_samples, len(test_images)))
    
    for img_path in samples:
        # Run inference
        results = model.predict(img_path, conf=0.25)
        
        # Plot results
        fig, ax = plt.subplots(figsize=(12, 10))
        ax.imshow(results[0].plot())
        plt.title(f"Inference on {os.path.basename(img_path)}")
        plt.axis('off')
        plt.show()
        
        # Print detections
        print(f"\nDetections for {os.path.basename(img_path)}:")
        boxes = results[0].boxes
        for i, box in enumerate(boxes):
            cls = int(box.cls[0])
            cls_name = model.names[cls]
            conf = box.conf[0].item()
            print(f"  {i+1}. Class: {cls_name}, Confidence: {conf:.4f}")

# Test the model on sample images if available
if 'model' in locals():
    test_img_dir = os.path.join(dataset_path, 'test', 'images')
    test_inference(model, test_img_dir, num_samples=3)

In [ ]:
# Export and save model
def export_model(model, format='onnx'):
    try:
        # Export the model in the specified format
        model.export(format=format)
        print(f"Model exported in {format.upper()} format successfully!")
    except Exception as e:
        print(f"Error exporting model: {e}")

# Export the model if available
if 'model' in locals():
    export_model(model)